In [ ]:
from datasets import load_dataset
from pathlib import Path
import re

DATA_FILE = "./data.txt"
DATASET_ID = "karpathy/fineweb-edu-100b-shuffle"

TAKE_N_DOCS = 1_000_000
MIN_CHARS = 400
MAX_CHARS = 8000
FLUSH_EVERY = 2000

SEP = "<|eos|>"

BAD_SUBSTR = [
    "cookie policy", "privacy policy", "terms of service",
    "all rights reserved", "subscribe to our newsletter",
    "accept cookies", "javascript required",
]
url_re = re.compile(r"https?://|www\.")
repeat_re = re.compile(r"(.)\1{10,}")


def clean_text(t: str) -> str:
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = t.strip()
    return t


def is_good_text(t: str) -> bool:
    if not t:
        return False
    t = t.strip()
    if len(t) < MIN_CHARS:
        return False
    if len(t) > MAX_CHARS:
        return False

    low = t.lower()
    if any(s in low for s in BAD_SUBSTR):
        return False

    if len(url_re.findall(low)) >= 3:
        return False

    if repeat_re.search(t):
        return False

    return True


def build_data_txt(out_path: str = DATA_FILE):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[SKIP] {out_path} already exists and is not empty.")
        return

    print("[INFO] streaming dataset...")
    stream = load_dataset(DATASET_ID, split="train", streaming=True)

    written = 0
    buffer_lines = []

    with out_path.open("w", encoding="utf-8") as f:
        for ex in stream:
            t = clean_text(ex.get("text", ""))
            if not is_good_text(t):
                continue

            buffer_lines.append(t)
            buffer_lines.append(SEP)

            written += 1

            if written % FLUSH_EVERY == 0:
                f.write("\n".join(buffer_lines) + "\n")
                buffer_lines = []
                print(f"[WRITE] docs={written}")

            if written >= TAKE_N_DOCS:
                break

        if buffer_lines:
            f.write("\n".join(buffer_lines) + "\n")

    print(f"[DONE] wrote {written} docs to {out_path}")


build_data_txt(DATA_FILE)

In [ ]:
!du -sh data.txt
